# 05 · Hybrid Search
### Practical LLM Engineering — Module 03: Embeddings & Retrieval

> **Objectives**
> - Why dense-only retrieval fails for keyword-heavy queries
> - BM25: the gold standard sparse retrieval algorithm
> - Reciprocal Rank Fusion (RRF) for combining dense + sparse results
> - Weighted hybrid scoring
> - Re-ranking with a cross-encoder
> - Engineering patterns: two-stage retrieval, score normalisation

---


## 1. Overview

**Dense retrieval** (embedding similarity) excels at semantic matching:
- "fast cars" ↔ "high-speed vehicles" ← finds this ✅
- "GPT-4 API rate limit" ← misses exact terms ❌

**Sparse retrieval** (BM25, TF-IDF) excels at exact term matching:
- "GPT-4 API rate limit" ← finds exact matches ✅
- "fast cars" ↔ "high-speed vehicles" ← misses paraphrase ❌

**Hybrid search** combines both:
- Semantic understanding from dense
- Keyword precision from sparse
- Consistently best retrieval quality across query types


## 2. Setup

In [ ]:
# !pip install rank-bm25 numpy matplotlib scikit-learn -q
import re, math, time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import normalize

print("✅ Libraries ready")

class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim   = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        return v / max(np.linalg.norm(v), 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts]))

embedder = MockEmbedder(dim=64)


## 3. BM25: Sparse Retrieval

BM25 (Best Match 25) is a probabilistic TF-IDF variant. For query term $q_i$ in document $d$:

$$
\text{BM25}(d, q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, d) \cdot (k_1 + 1)}{f(q_i, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
$$

where:
- $f(q_i, d)$ — term frequency of $q_i$ in $d$
- $|d|$ — document length in tokens
- $\text{avgdl}$ — average document length in corpus
- $k_1 \in [1.2, 2.0]$ — term saturation parameter
- $b \in [0, 1]$ — length normalisation ($b=0.75$ standard)


In [ ]:
class BM25:
    """
    BM25 implementation from scratch.
    Parameters: k1=1.5, b=0.75 (standard defaults).
    """
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self._corpus:    list[list[str]] = []
        self._doc_freqs: dict[str, int]  = defaultdict(int)
        self._avgdl      = 0.0
        self._N          = 0

    def _tokenise(self, text: str) -> list[str]:
        return re.findall(r'\b\w+\b', text.lower())

    def fit(self, documents: list[str]):
        """Build the BM25 index from a corpus of documents."""
        self._corpus = [self._tokenise(d) for d in documents]
        self._N      = len(self._corpus)
        self._avgdl  = np.mean([len(d) for d in self._corpus])

        for doc in self._corpus:
            for term in set(doc):
                self._doc_freqs[term] += 1

    def _idf(self, term: str) -> float:
        n    = self._doc_freqs.get(term, 0)
        return math.log((self._N - n + 0.5) / (n + 0.5) + 1)

    def score(self, query: str, doc_idx: int) -> float:
        query_terms = self._tokenise(query)
        doc         = self._corpus[doc_idx]
        dl          = len(doc)
        tf_map      = defaultdict(int)
        for t in doc: tf_map[t] += 1

        score = 0.0
        for term in query_terms:
            if term not in tf_map:
                continue
            tf  = tf_map[term]
            idf = self._idf(term)
            score += idf * (tf * (self.k1 + 1)) / (
                tf + self.k1 * (1 - self.b + self.b * dl / self._avgdl)
            )
        return score

    def get_scores(self, query: str) -> np.ndarray:
        return np.array([self.score(query, i) for i in range(self._N)])

    def get_top_k(self, query: str, k: int = 5) -> list[tuple[int, float]]:
        scores  = self.get_scores(query)
        top_idx = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in top_idx]


# ── Demo corpus ───────────────────────────────────────────────────────
CORPUS = [
    "FAISS provides efficient GPU-accelerated similarity search for billion-scale vectors.",
    "BM25 is the standard sparse retrieval algorithm used in Elasticsearch and Solr.",
    "Transformer models use multi-head self-attention to process sequences in parallel.",
    "Hybrid search combines dense vector retrieval with sparse BM25 keyword matching.",
    "GPT-4 API rate limits are enforced per minute and per day for each API key.",
    "The Eiffel Tower was built between 1887 and 1889 for the World Fair in Paris.",
    "Reciprocal Rank Fusion merges ranked lists from multiple retrieval systems.",
    "Product quantisation compresses high-dimensional vectors to reduce memory usage.",
    "Rome was founded on the banks of the Tiber River in central Italy.",
    "HNSW builds a hierarchical proximity graph for approximate nearest-neighbour search.",
]

bm25 = BM25()
bm25.fit(CORPUS)

# Test queries
queries = [
    "GPT-4 API rate limit per minute",   # exact keyword query
    "how does attention work in neural networks",  # semantic query
    "vector similarity search algorithms",  # mixed
]

print("BM25 results:")
for q in queries:
    top = bm25.get_top_k(q, k=3)
    print(f"\\nQ: {q}")
    for idx, score in top:
        print(f"  [{score:.3f}] {CORPUS[idx][:65]}")


## 4. Reciprocal Rank Fusion (RRF)

RRF (Cormack et al., 2009) merges two ranked lists without needing score normalisation:

$$
\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}
$$

where $k = 60$ is a smoothing constant and $\text{rank}_r(d)$ is the rank of document $d$ in ranked list $r$.

RRF is robust because:
- It is insensitive to the absolute scale of scores (no normalisation needed)
- Documents ranked high in either list get a boost
- The $k=60$ constant limits the advantage of rank-1 documents


In [ ]:
def rrf_merge(ranked_lists: list[list[int]], k: int = 60) -> list[tuple[int, float]]:
    """
    Reciprocal Rank Fusion over multiple ranked lists.
    ranked_lists: list of doc-id lists, each sorted best-first.
    Returns: sorted list of (doc_id, rrf_score).
    """
    scores: dict[int, float] = defaultdict(float)
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def dense_search(query: str, corpus: list[str], embedder,
                  k: int = 10) -> list[tuple[int, float]]:
    """Embed-based cosine similarity search."""
    q_vec    = embedder.embed([query])[0]
    doc_vecs = embedder.embed(corpus)
    scores   = doc_vecs @ q_vec
    top_idx  = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top_idx]


def hybrid_rrf(query: str, corpus: list[str],
               bm25_index: BM25, embedder,
               k_dense: int = 10, k_sparse: int = 10,
               rrf_k: int = 60) -> list[tuple[int, float]]:
    """Hybrid RRF: combine dense + sparse ranked lists."""
    dense_ids  = [i for i, _ in dense_search(query, corpus, embedder, k_dense)]
    sparse_ids = [i for i, _ in bm25_index.get_top_k(query, k_sparse)]
    return rrf_merge([dense_ids, sparse_ids], k=rrf_k)


# ── Compare retrieval methods ─────────────────────────────────────────
EVAL_QUERIES = [
    ("GPT-4 API rate limit per key",      {4}),          # exact keyword
    ("attention mechanism transformers",   {2}),          # semantic
    ("ANN similarity search algorithms",  {0, 7, 9}),    # mixed
    ("how to fuse ranked retrieval lists", {3, 6}),       # conceptual
]

methods = {
    "Dense only":   lambda q: [i for i, _ in dense_search(q, CORPUS, embedder, 5)],
    "BM25 only":    lambda q: [i for i, _ in bm25.get_top_k(q, 5)],
    "Hybrid RRF":   lambda q: [i for i, _ in hybrid_rrf(q, CORPUS, bm25, embedder)[:5]],
}

print("Recall@3 comparison:")
print(f"{'Query':<40} {'Dense':>7} {'BM25':>7} {'Hybrid':>8}")
print("-" * 65)

for q_text, relevant in EVAL_QUERIES:
    row = f"{q_text[:38]:<40}"
    for name, fn in methods.items():
        retrieved = fn(q_text)
        recall    = len(set(retrieved[:3]) & relevant) / len(relevant)
        row      += f"  {recall:>5.2f}"
    print(row)


## 5. Weighted Hybrid Scoring

In [ ]:
def normalise_scores(scores: list[tuple[int, float]]) -> list[tuple[int, float]]:
    """Min-max normalise scores to [0, 1]."""
    if not scores: return []
    vals  = np.array([s for _, s in scores])
    lo, hi = vals.min(), vals.max()
    if hi == lo: return [(i, 1.0) for i, _ in scores]
    normed = (vals - lo) / (hi - lo)
    return [(idx, float(n)) for (idx, _), n in zip(scores, normed)]


def hybrid_weighted(query: str, corpus: list[str],
                     bm25_index: BM25, embedder,
                     alpha: float = 0.5,
                     k: int = 10) -> list[tuple[int, float]]:
    """
    Weighted hybrid: score = alpha * dense_norm + (1-alpha) * bm25_norm.
    alpha=1.0 → dense only; alpha=0.0 → BM25 only.
    """
    dense_raw  = dense_search(query, corpus, embedder, len(corpus))
    sparse_raw = [(i, s) for i, s in enumerate(bm25_index.get_scores(query))]

    dense_norm  = dict(normalise_scores(dense_raw))
    sparse_norm = dict(normalise_scores(sparse_raw))

    all_ids = set(dense_norm) | set(sparse_norm)
    combined = {
        i: alpha * dense_norm.get(i, 0.0) + (1 - alpha) * sparse_norm.get(i, 0.0)
        for i in all_ids
    }
    return sorted(combined.items(), key=lambda x: -x[1])[:k]


# ── Alpha sweep ───────────────────────────────────────────────────────
alphas = np.linspace(0, 1, 11)

recall_by_alpha = []
for alpha in alphas:
    recalls = []
    for q_text, relevant in EVAL_QUERIES:
        retrieved = [i for i, _ in hybrid_weighted(q_text, CORPUS, bm25, embedder, alpha, k=5)[:3]]
        recalls.append(len(set(retrieved) & relevant) / len(relevant))
    recall_by_alpha.append(np.mean(recalls))

best_alpha = alphas[np.argmax(recall_by_alpha)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, recall_by_alpha, "o-", color="#3498db", lw=2, markersize=7)
ax.axvline(best_alpha, color="#e74c3c", linestyle="--", lw=1.5,
           label=f"Best α={best_alpha:.1f} (recall={max(recall_by_alpha):.3f})")
ax.set_xlabel("α  (0 = BM25 only,  1 = dense only)")
ax.set_ylabel("Mean Recall@3")
ax.set_title("Weighted Hybrid Search: α sweep over 4 query types")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Optimal alpha: {best_alpha:.1f}")


## 6. Cross-Encoder Re-ranking

**Two-stage retrieval** decouples speed from quality:

```
Stage 1 — Bi-encoder recall (fast):
  Embed query + docs independently → ANN search → top-100 candidates

Stage 2 — Cross-encoder rerank (accurate, slow):
  Concatenate [query, doc] → encoder → relevance score
  Sort top-100 → return top-5
```

Cross-encoders are more accurate because they see both query and document together,
allowing full attention-based interaction. They are too slow to run over the full corpus
(no pre-computed embeddings) but fast enough for 50–200 candidates.


In [ ]:
# ── Cross-encoder simulation ──────────────────────────────────────────
class MockCrossEncoder:
    """
    Simulates a cross-encoder by measuring term overlap + embedding sim.
    Replace with: from sentence_transformers import CrossEncoder
                  model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
    """
    def __init__(self, embedder):
        self.embedder = embedder

    def predict(self, pairs: list[tuple[str, str]]) -> np.ndarray:
        scores = []
        for q, doc in pairs:
            # Keyword overlap
            q_terms   = set(q.lower().split())
            doc_terms = set(doc.lower().split())
            overlap   = len(q_terms & doc_terms) / max(len(q_terms), 1)
            # Embedding similarity
            vecs  = self.embedder.embed([q, doc])
            emb_s = float(np.dot(vecs[0], vecs[1]))
            scores.append(0.4 * overlap + 0.6 * emb_s)
        return np.array(scores)


def two_stage_retrieval(query: str, corpus: list[str],
                         bm25_index: BM25, embedder,
                         cross_encoder,
                         stage1_k: int = 10,
                         stage2_k: int = 3) -> list[dict]:
    """
    Two-stage retrieval: hybrid recall → cross-encoder rerank.
    """
    # Stage 1: Hybrid recall (fast)
    stage1 = hybrid_rrf(query, corpus, bm25_index, embedder, stage1_k, stage1_k)
    candidates = [i for i, _ in stage1[:stage1_k]]

    # Stage 2: Cross-encoder rerank (accurate)
    pairs  = [(query, corpus[i]) for i in candidates]
    scores = cross_encoder.predict(pairs)

    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    return [
        {"id": idx, "score": float(sc), "text": corpus[idx]}
        for idx, sc in ranked[:stage2_k]
    ]


cross_encoder = MockCrossEncoder(embedder)

print("Two-stage retrieval (hybrid recall → cross-encoder rerank):\n")
for q_text, relevant in EVAL_QUERIES:
    results = two_stage_retrieval(q_text, CORPUS, bm25, embedder, cross_encoder)
    retrieved = {r["id"] for r in results}
    recall    = len(retrieved & relevant) / len(relevant)
    print(f"Q: {q_text[:55]}")
    for r in results:
        hit = "✅" if r["id"] in relevant else "  "
        print(f"  {hit} [{r['score']:.3f}] {r['text'][:60]}")
    print(f"  Recall@3 = {recall:.2f}\n")


## 7. Engineering Insights

In [ ]:
# ── Full pipeline latency breakdown ──────────────────────────────────
import time

query = "vector similarity search for large scale retrieval"
N_corpus = len(CORPUS)

# Stage 1: dense
t0 = time.perf_counter()
for _ in range(100): dense_search(query, CORPUS, embedder, 10)
dense_ms = (time.perf_counter() - t0) * 10

# Stage 1: BM25
t0 = time.perf_counter()
for _ in range(100): bm25.get_top_k(query, 10)
bm25_ms = (time.perf_counter() - t0) * 10

# Stage 1: hybrid RRF
t0 = time.perf_counter()
for _ in range(100): hybrid_rrf(query, CORPUS, bm25, embedder)
hybrid_ms = (time.perf_counter() - t0) * 10

# Stage 2: cross-encoder (10 candidates)
t0 = time.perf_counter()
for _ in range(100):
    pairs = [(query, CORPUS[i]) for i in range(10)]
    cross_encoder.predict(pairs)
rerank_ms = (time.perf_counter() - t0) * 10

print("Latency breakdown (10-doc corpus, 100 iterations avg):")
print(f"  Dense embed + search   : {dense_ms:.2f}ms")
print(f"  BM25 scoring           : {bm25_ms:.2f}ms")
print(f"  Hybrid RRF (both)      : {hybrid_ms:.2f}ms")
print(f"  Cross-encoder (10 cand): {rerank_ms:.2f}ms")
print(f"  Full two-stage total   : {hybrid_ms+rerank_ms:.2f}ms")


## 8. Key Takeaways

| Concept | Summary |
|---|---|
| **BM25** | Sparse retrieval — excels at exact keyword matching |
| **Dense retrieval** | Semantic embedding — excels at paraphrase and concept matching |
| **Hybrid search** | Combines both — consistently best quality |
| **RRF** | Score-free rank fusion; robust and simple |
| **Weighted hybrid** | Tune α for your query distribution (α ≈ 0.5 is a good default) |
| **Cross-encoder** | Accurate reranker; use in two-stage pipeline (top-100 → top-5) |
| **Two-stage** | Stage 1: fast recall; Stage 2: accurate rerank |
| **Alpha tuning** | Measure on a dev set; keyword-heavy domains prefer lower α |
